# Budgets and failure

Every agent turn is an LLM call, and an agent decides for itself how many turns it takes. That is the point — and it is also why you want limits you can prove will hold.

This notebook is about the unhappy paths: runs that cost too much, loop too long, return the wrong shape, or simply cannot be done. Each limit below is set small enough that you will *see* it fire.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [1]:
%pip install -q jaz-lang

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [3]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## Capping cost across a whole tree

`BudgetPool` is a shared allowance. Everything inside the `with` block draws from it — the top-level agent and every subagent — so one pool bounds the entire run, not each call separately.

The budget below is a thousandth of a cent, which one call already exceeds:

In [4]:
from jaz import invoke
from jaz.hooks import ReturnType, BudgetPool
from jaz.exceptions import BudgetPoolExhaustedError

# Deliberately far too small: one LLM call costs more than this.
try:
    with BudgetPool(cost_budget=0.00001):
        invoke(
            ReturnType(list),
            task="Write a detailed 10-step plan for migrating a database.",
        )
except BudgetPoolExhaustedError as exc:
    print("stopped:", type(exc).__name__)
    print(str(exc)[:220])

stopped: BudgetPoolExhaustedError
LLM cost budget exhausted: $0.001207 spent, budget was $0.000010


The same pool can cap the number of LLM calls instead of, or as well as, the spend:

In [5]:
from jaz.exceptions import JazException

# The same pool also caps how many LLM calls the whole tree may make.
try:
    with BudgetPool(calls_budget=2):
        invoke(
            ReturnType(int),
            task=(
                "Count from 1 to 5, printing each number in a separate REPL iteration, "
                "then return 5."
            ),
        )
except JazException as exc:
    print("stopped:", type(exc).__name__)

stopped: BudgetPoolExhaustedError


## Capping turns per agent

`BudgetPool` bounds the tree. `IterationLimit` bounds a single agent's REPL loop — how many turns it may take before it must return.

Two turns, for a task that needs more:

In [6]:
from jaz.hooks import IterationLimit
from jaz.exceptions import IterationLimitExhaustedError

try:
    with IterationLimit(max_iterations=2):
        invoke(
            ReturnType(int),
            task=(
                "Print the numbers 1, 2, 3, 4, 5 one per REPL iteration, each in its own "
                "turn, then return 5."
            ),
        )
except IterationLimitExhaustedError as exc:
    print("stopped:", type(exc).__name__)
    print(str(exc)[:200])

stopped: IterationLimitExhaustedError
No return value after 2 REPL iterations (depth 1)


## When the answer is the wrong shape

A limit stops a run that costs too much. `ValidateReturn` handles the other failure: a run that finishes and hands back something wrong.

Your check runs after the agent returns. If it raises, the message goes back to the agent and it tries again — so you can see the retry happen:

In [7]:
from jaz.hooks import ValidateReturn

seen = []


def rejected_once(result):
    """Contrived on purpose: rejects the first answer whatever it is.

    Real validators check something you care about — a schema, a row count, an invariant.
    This one just proves the retry actually happens.
    """
    seen.append(result)
    assert len(seen) > 1, "first answer is always rejected; please return it again"


answer = invoke(
    ReturnType(int),
    ValidateReturn(rejected_once),
    task="Return the number 7.",
)

print(f"attempts: {len(seen)} -> {seen}")
print(f"final answer: {answer}")

attempts: 2 -> [7, 7]
final answer: 7


Two attempts, one answer. In real code the validator asserts something you actually care about — the row count, a schema, an invariant — and the agent gets your assertion message as the reason to try again.

## Catching failures

There are two kinds, and they reach you differently.

**JAZ stopped the run** — a budget, a limit, a return-type violation. Everything JAZ raises descends from `jaz.exceptions.JazException`, so one `except` covers all of it.

**The agent gave up** — asked for something impossible, it raises, and *its* exception propagates to you unchanged. That is not a JAZ error, so a bare `except JazException` will not catch it:

In [8]:
def run_safely(task, **inputs):
    """Two different kinds of failure, and they arrive differently."""
    try:
        return invoke(ReturnType(int), task=task, **inputs)
    except JazException as exc:
        # JAZ stopped the run: a budget, a limit, a return-type violation.
        print(f"  JAZ stopped it -> {type(exc).__name__}")
        return None
    except Exception as exc:
        # The agent decided the task was impossible and raised. Its exception is yours.
        print(f"  the agent raised -> {type(exc).__name__}: {str(exc)[:90]}")
        return None


print("a task it can do:")
print(" ", run_safely("Return the number 3."))

print("a task it cannot:")
print(" ", run_safely("Return today's gold price in USD. Do not guess or invent a number."))

print("a budget too small:")
try:
    with BudgetPool(cost_budget=0.00001):
        run_safely("Write a 10-step migration plan and return its length.")
except JazException as exc:
    print(f"  JAZ stopped it -> {type(exc).__name__}")

a task it can do:


  3
a task it cannot:


  the agent raised -> Exception: Cannot retrieve today's gold price in USD in this environment (no web/API access and no ab
  None
a budget too small:


## What to take away

- **`BudgetPool` bounds a whole tree**, by spend or by call count.
- **`IterationLimit` bounds one agent's loop.**
- **`ValidateReturn` checks the result** and sends the agent back with your message when the check fails.
- **`JazException` is the base class** for everything JAZ raises — catch it.
- Set limits deliberately. Unset, recursion and iteration are unbounded.

---

**Next:** [Hooks and tracing](05-hooks-and-tracing.ipynb) — every limit in this notebook is a hook, and you can write your own.